# 🎓 Treinamento Neural Oficial da Voz do Mascote FEFO (Coqui XTTS-v2)

**Fine-Tuning com 577 Áudios Oficiais do FEFO**

Neste notebook, vamos treinar de verdade a camada neural do modelo **XTTS-v2** com a biblioteca completa de áudios e transcrições do FEFO. O modelo aprenderá a anatomia vocal, o tom infantil e alegre, a respiração e a dicção natural do mascote em Português do Brasil, eliminando as falhas de pronúncia e sotaques.

⏱️ **Tempo estimado:** ~15 a 20 minutos na GPU T4 gratuita do Google Colab.


In [ ]:
# 1. Instalação do Ambiente de Treinamento e Dependências
import os, torch
os.environ['COQUI_TOS_AGREED'] = '1'

print('=' * 60)
if torch.cuda.is_available():
    print(f'✓ GPU DETECTADA: {torch.cuda.get_device_name(0)}')
    print(f'✓ VRAM Disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ ATENÇÃO: Nenhuma GPU detectada! Ative em: Ambiente de execução -> Alterar tipo -> T4 GPU')
print('=' * 60)

print('[*] Instalando dependências de sistema e compiladores de áudio...')
!sudo apt-get update -qq && sudo apt-get install -y -qq espeak-ng

print('[*] Instalando pacotes Coqui TTS e Trainer...')
!pip install --upgrade coqui-tts coqui-tts-trainer accelerate gradio soundfile pydub

print('✓ Ambiente configurado com sucesso!')


In [ ]:
# 2. Baixar Repositório e Descompactar Dataset Oficial do FEFO (577 Áudios)
import os, zipfile, glob

%cd /content
if not os.path.exists('/content/fefo-repo'):
    print('[*] Clonando repositório oficial do FEFO...')
    !git clone https://github.com/elCortelini/fefo-v1.git /content/fefo-repo
else:
    print('[*] Atualizando repositório do FEFO...')
    !cd /content/fefo-repo && git pull

dataset_zip = '/content/fefo-repo/fefo_training_dataset.zip'
dataset_dir = '/content/fefo_dataset'
os.makedirs(dataset_dir, exist_ok=True)

print('[*] Descompactando 577 áudios e transcrições do FEFO...')
with zipfile.ZipFile(dataset_zip, 'r') as z:
    z.extractall(dataset_dir)

total_wavs = len(glob.glob(f'{dataset_dir}/wavs/*.wav'))
meta_file = f'{dataset_dir}/metadata.csv'

print('=' * 60)
print(f'✓ Dataset do FEFO pronto!')
print(f'✓ Total de áudios: {total_wavs} arquivos WAV')
print(f'✓ Arquivo de metadados: {meta_file} (Existe: {os.path.exists(meta_file)})')
print('=' * 60)


In [ ]:
# 3. Baixar Pesos Base Oficiais do Modelo XTTS-v2
import os, urllib.request

base_model_dir = '/content/base_xtts_v2'
os.makedirs(base_model_dir, exist_ok=True)

hf_files = {
    'model.pth': 'https://huggingface.co/coqui/XTTS-v2/resolve/main/model.pth',
    'config.json': 'https://huggingface.co/coqui/XTTS-v2/resolve/main/config.json',
    'vocab.json': 'https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json',
    'dvae.pth': 'https://huggingface.co/coqui/XTTS-v2/resolve/main/dvae.pth',
    'mel_stats.pth': 'https://huggingface.co/coqui/XTTS-v2/resolve/main/mel_stats.pth',
}

print('[*] Verificando e baixando pesos do modelo base XTTS-v2...')
for fname, url in hf_files.items():
    dest = os.path.join(base_model_dir, fname)
    if not os.path.exists(dest) or os.path.getsize(dest) < 1000:
        print(f'   Baixando {fname}...')
        !wget -q -O {dest} {url}
    print(f'   ✓ {fname} pronto ({os.path.getsize(dest) / (1024*1024):.1f} MB)')

print('=' * 60)
print('✓ Todos os pesos base carregados e prontos para o treinamento!')
print('=' * 60)


In [ ]:
# 4. Execução do Treinamento Neural (Fine-Tuning) do FEFO
import os, sys, torch
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import (
    GPTArgs,
    GPTTrainer,
    GPTTrainerConfig,
    XttsAudioConfig
)

output_train_dir = '/content/fefo_treinado'
os.makedirs(output_train_dir, exist_ok=True)

# Configuração do Dataset FEFO
config_dataset = BaseDatasetConfig(
    formatter='ljspeech',
    dataset_name='fefo_voice',
    path='/content/fefo_dataset/',
    meta_file_train='/content/fefo_dataset/metadata.csv',
    language='pt'
)

# Áudio de referência oficial
ref_speaker = '/content/fefo-repo/repository/ficofelizporvc.wav'
if not os.path.exists(ref_speaker):
    ref_speaker = '/content/fefo_dataset/wavs/fefo_0001.wav'

model_args = GPTArgs(
    max_conditioning_length=132300,  # 6 segundos
    min_conditioning_length=66150,   # 3 segundos
    debug_loading_failures=False,
    max_wav_length=255995,
    max_text_length=200,
    mel_norm_file='/content/base_xtts_v2/mel_stats.pth',
    dvae_checkpoint='/content/base_xtts_v2/dvae.pth',
    xtts_checkpoint='/content/base_xtts_v2/model.pth',
    tokenizer_file='/content/base_xtts_v2/vocab.json',
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)

audio_config = XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000)

# Hiperparâmetros otimizados para GPU T4 do Colab
NUM_EPOCHS = 6
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 64

config = GPTTrainerConfig(
    output_path=output_train_dir,
    model_args=model_args,
    run_name='fefo_xtts_finetune',
    project_name='FEFO_Voice_AI',
    run_description='Treinamento da Voz do Mascote FEFO',
    audio=audio_config,
    batch_size=BATCH_SIZE,
    batch_group_size=48,
    eval_batch_size=BATCH_SIZE,
    num_loader_workers=4,
    eval_split_max_size=32,
    print_step=25,
    plot_step=100,
    log_model_step=500,
    save_step=1000,
    save_n_checkpoints=2,
    save_checkpoints=True,
    epochs=NUM_EPOCHS,
    lr=5e-6,
    optimizer_wd_only_on_weights=True,
    grad_accum_steps=GRAD_ACCUM_STEPS,
)

print('=' * 60)
print(f'🚀 INICIANDO TREINAMENTO NEURAL DO FEFO ({NUM_EPOCHS} ÉPOCAS)...')
print('⏱️ O processo levará cerca de 15 a 20 minutos na GPU T4.')
print('=' * 60)

# Carregar amostras e inicializar modelo e trainer
train_samples, eval_samples = load_tts_samples(
    config_dataset,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)

model = GPTTrainer.init_from_config(config)

trainer = Trainer(
    TrainerArgs(
        restore_path=None,
        skip_train_epoch=False,
        start_with_eval=False,
        grad_accum_steps=config.grad_accum_steps,
    ),
    config,
    output_path=output_train_dir,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

# Roda o treinamento
trainer.fit()

print('=' * 60)
print('🎉 TREINAMENTO DO FEFO CONCLUÍDO COM SUCESSO!')
print('=' * 60)


In [ ]:
# 5. Teste Imediato com a Voz Treinada do FEFO (Ouvir a Diferença)
import os, glob, shutil, IPython.display as ipd
from TTS.api import TTS

# Localizar o checkpoint treinado
checkpoints = glob.glob('/content/fefo_treinado/**/checkpoint_*.pth', recursive=True)
best_models = glob.glob('/content/fefo_treinado/**/best_model.pth', recursive=True)

trained_checkpoint = best_models[0] if best_models else (checkpoints[-1] if checkpoints else None)

if not trained_checkpoint:
    print('Aviso: checkpoint não localizado na pasta padrão. Usando base.')
    trained_checkpoint = '/content/base_xtts_v2/model.pth'

print(f'✓ Checkpoint treinado carregado: {trained_checkpoint}')

# Copiar arquivos de configuração auxiliares para a pasta do modelo
final_model_dir = '/content/fefo_modelo_final'
os.makedirs(final_model_dir, exist_ok=True)
shutil.copy(trained_checkpoint, os.path.join(final_model_dir, 'model.pth'))
shutil.copy('/content/base_xtts_v2/vocab.json', os.path.join(final_model_dir, 'vocab.json'))
shutil.copy('/content/base_xtts_v2/config.json', os.path.join(final_model_dir, 'config.json'))

# Carrega o modelo XTTS com os pesos treinados do FEFO
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts_fefo = TTS(model_path=final_model_dir, config_path=os.path.join(final_model_dir, 'config.json')).to(device)

# Áudio de referência de suporte
ref_audio = '/content/fefo-repo/repository/ficofelizporvc.wav'
if not os.path.exists(ref_audio):
    ref_audio = '/content/fefo_dataset/wavs/fefo_0001.wav'

frase_teste = 'Olá amiguinho! Eu sou o Fefo, agora minha voz é treinada de verdade nos neurônios! Que alegria estar com você!'
saida_teste = '/content/teste_fefo_treinado.wav'

print('[*] Sintetizando fala com os neurônios treinados do FEFO...')
tts_fefo.tts_to_file(
    text=frase_teste,
    file_path=saida_teste,
    speaker_wav=ref_audio,
    language='pt',
    temperature=0.45,
    repetition_penalty=2.0
)

print('=' * 60)
print('🎉 ÁUDIO DO FEFO TREINADO! CLIQUE NO PLAY ABAIXO PARA OUVIR:')
print('=' * 60)
display(ipd.Audio(saida_teste))


In [ ]:
# 6. Compactar Modelo Treinado e Salvar no Google Drive
import os, zipfile
from google.colab import files

zip_saida = '/content/fefo_modelo_xtts_v2_treinado.zip'
print('[*] Compactando modelo treinado do FEFO...')

with zipfile.ZipFile(zip_saida, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(final_model_dir):
        fp = os.path.join(final_model_dir, f)
        z.write(fp, arcname=f)

print(f'✓ Modelo compactado com sucesso! Tamanho: {os.path.getsize(zip_saida) / (1024*1024):.1f} MB')
print('💡 Dica: Você pode baixar este arquivo ou conectar o Google Drive abaixo para salvar para sempre!')

# Opcional: Salvar no Google Drive com 1 comando
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(zip_saida, '/content/drive/MyDrive/fefo_modelo_xtts_v2_treinado.zip')


In [ ]:
# 7. Iniciar WebUI e API REST com o Modelo Treinado do FEFO
import os, base64, gradio as gr
from fastapi import Request
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware

def gerar_fefo_treinado(texto, velocidade=1.0, temperatura=0.45):
    if not texto or not texto.strip():
        return None
    out_path = '/content/fefo_api_saida.wav'
    tts_fefo.tts_to_file(
        text=texto.strip(),
        file_path=out_path,
        speaker_wav=ref_audio,
        language='pt',
        speed=float(velocidade),
        temperature=float(temperatura),
        repetition_penalty=2.0
    )
    return out_path

with gr.Blocks(title='🧸 FEFO - Voz Treinada com IA') as app:
    gr.Markdown('# 🎙️ Mascote FEFO - Modelo Neural Treinado')
    gr.Markdown('Voz afinada diretamente nos neurônios do XTTS-v2 com 577 áudios oficiais.')
    
    with gr.Row():
        with gr.Column():
            txt_in = gr.Textbox(label='Texto do FEFO:', value='Oi amiguinho! Vamos brincar juntos?', lines=3)
            spd_in = gr.Slider(0.7, 1.4, value=1.0, step=0.05, label='Velocidade')
            temp_in = gr.Slider(0.1, 0.9, value=0.45, step=0.05, label='Expressividade')
            btn = gr.Button('✨ Gerar Voz do FEFO Treinado', variant='primary')
        with gr.Column():
            aud_out = gr.Audio(label='Áudio Gerado', type='filepath')
            
    btn.click(fn=gerar_fefo_treinado, inputs=[txt_in, spd_in, temp_in], outputs=[aud_out])

fastapi_app = app.app
fastapi_app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

@fastapi_app.get('/health')
def health():
    return {'ok': True, 'status': 'online', 'model': 'XTTS-v2-FEFO-TRAINED', 'trained': True}

@fastapi_app.post('/api/generate')
async def api_generate(req: Request):
    try:
        b = await req.json()
        txt = b.get('text', '').strip()
        if not txt: return JSONResponse(status_code=400, content={'ok': False, 'message': 'Texto vazio'})
        spd = float(b.get('speed', 1.0))
        tmp = float(b.get('temperature', 0.45))
        wav = gerar_fefo_treinado(txt, spd, tmp)
        with open(wav, 'rb') as f: data = f.read()
        return {'ok': True, 'audio_base64': base64.b64encode(data).decode('utf-8'), 'engine': 'XTTS-v2-FEFO-TRAINED'}
    except Exception as e:
        return JSONResponse(status_code=500, content={'ok': False, 'message': str(e)})

print('=' * 60)
print('🚀 INICIANDO API COM O MODELO TREINADO DO FEFO...')
print('💡 Copie o link .gradio.live e cole no Catalog Admin!')
print('=' * 60)
app.launch(share=True, debug=False)
